In [3]:
# ============ CELL 1: CONFIG ============
import os, re, json, math, random, warnings, hashlib
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

ROOT  = Path(r"F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To_Compare\CARE_To_Compare")
CACHE = ROOT.parent / "sagewt_cache"; CACHE.mkdir(exist_ok=True, parents=True)

FARMS = ["Wind Farm A", "Wind Farm B", "Wind Farm C"]
FARM_TAG = {"Wind Farm A": "A", "Wind Farm B": "B", "Wind Farm C": "C"}

# --- CARE conventions (confirm against the bundled README once) ---
NORMAL_STATUS = {0, 2}      # status_type_id treated as healthy operation for training/calibration
RESOLUTION_MIN = 10         # SCADA resolution
WINDOW = 144                # 24 h of context
HORIZON = 1                 # forecast next step

# --- Zenodo v6 known issues -> we trust Avg only ---
USE_STATS = ("avg",)        # set to ("avg","min","max","std") only for the ablation
SEED = 1337

random.seed(SEED); np.random.seed(SEED)
print("Root exists:", ROOT.exists())
print("Cache      :", CACHE)
for f in FARMS:
    d = ROOT / f / "datasets"
    print(f"{f:14s} exists={ (ROOT/f).exists() }  event files={ len(list(d.glob('*.csv'))) if d.exists() else 0 }")

Root exists: True
Cache      : F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To_Compare\sagewt_cache
Wind Farm A    exists=True  event files=22
Wind Farm B    exists=True  event files=15
Wind Farm C    exists=True  event files=58


In [4]:
# ============ CELL 2: SCHEMA PROBE ============
def sniff_sep(path, n=3):
    with open(path, "r", encoding="utf-8", errors="ignore") as fh:
        head = "".join([fh.readline() for _ in range(n)])
    return ";" if head.count(";") > head.count(",") else ","

def read_csv_auto(path, **kw):
    return pd.read_csv(path, sep=sniff_sep(path), **kw)

for f in FARMS:
    ei = read_csv_auto(ROOT / f / "event_info.csv")
    fd = read_csv_auto(ROOT / f / "feature_description.csv")
    print("=" * 70); print(f)
    print("  event_info cols   :", list(ei.columns))
    print("  event_info shape  :", ei.shape)
    print(ei.head(3).to_string())
    print("  feature_desc cols :", list(fd.columns), "| shape:", fd.shape)
    print(fd.head(5).to_string())

# one raw event file, header only
probe = sorted((ROOT / FARMS[0] / "datasets").glob("*.csv"))[0]
hdr = read_csv_auto(probe, nrows=5)
print("=" * 70); print("sample event:", probe.name, "| n_cols:", hdr.shape[1])
print("first 15 cols:", list(hdr.columns)[:15])
print(hdr.iloc[:3, :10].to_string())

Wind Farm A
  event_info cols   : ['asset', 'event_id', 'event_label', 'event_start', 'event_start_id', 'event_end', 'event_end_id', 'event_description']
  event_info shape  : (22, 8)
   asset  event_id event_label          event_start  event_start_id            event_end  event_end_id    event_description
0     11        68     anomaly  2023-07-28 13:20:00           52063  2023-08-11 13:10:00         54076  Transformer failure
1     21        22     anomaly  2023-08-12 09:50:00           51888  2023-08-19 10:00:00         52892      Hydraulic group
2     21        72     anomaly  2023-10-10 08:40:00           52497  2023-10-17 08:40:00         53505      Gearbox failure
  feature_desc cols : ['sensor_name', 'statistics_type', 'description', 'unit', 'is_angle', 'is_counter'] | shape: (54, 6)
    sensor_name                  statistics_type              description unit  is_angle  is_counter
0      sensor_0                          average      Ambient temperature   �C     False       F

In [5]:
# ============ CELL 3: EVENT INDEX ============
def resolve(cols, *cands, required=True, label=""):
    low = {c.lower().strip(): c for c in cols}
    for c in cands:
        if c in low: return low[c]
    for c in cands:                       # substring fallback
        for k, v in low.items():
            if c in k: return v
    if required: raise KeyError(f"[{label}] none of {cands} in {list(cols)[:25]}")
    return None

def load_event_info(farm):
    ei = read_csv_auto(ROOT / farm / "event_info.csv")
    c = ei.columns
    m = dict(
        event_id   = resolve(c, "event_id", "id", label="event_id"),
        label      = resolve(c, "event_label", "label", label="label"),
        asset_id   = resolve(c, "asset_id", "event_asset_id", "asset", label="asset_id"),
        start      = resolve(c, "event_start", "start_time", "start", label="start"),
        end        = resolve(c, "event_end", "end_time", "end", label="end"),
        descr      = resolve(c, "event_description", "description", required=False),
        start_id   = resolve(c, "event_start_id", required=False),
        end_id     = resolve(c, "event_end_id", required=False),
    )
    out = pd.DataFrame({k: ei[v] for k, v in m.items() if v is not None})
    out["farm"] = FARM_TAG[farm]
    out["label"] = out["label"].astype(str).str.strip().str.lower()
    for col in ("start", "end"):
        out[col] = pd.to_datetime(out[col], errors="coerce")
    out["asset_uid"] = out["farm"] + "_" + out["asset_id"].astype(str)
    out["uid"] = out["farm"] + "_e" + out["event_id"].astype(str)
    out["path"] = [ROOT / farm / "datasets" / f"{e}.csv" for e in out["event_id"]]
    out["file_ok"] = [p.exists() for p in out["path"]]
    return out

EVENTS = pd.concat([load_event_info(f) for f in FARMS], ignore_index=True)
EVENTS["is_anomaly"] = (EVENTS["label"] == "anomaly").astype(int)
print(EVENTS.shape)
print(EVENTS.groupby(["farm", "label"]).size().unstack(fill_value=0))
print("missing files:", (~EVENTS.file_ok).sum())
EVENTS.head()

(95, 14)
label  anomaly  normal
farm                  
A           12      10
B            6       9
C           27      31
missing files: 0


,event_id,label,asset_id,start,end,descr,start_id,end_id,farm,asset_uid,uid,path,file_ok,is_anomaly
0,68,anomaly,11,2023-07-28 13:20:00,2023-08-11 13:10:00,Transformer failure,52063,54076,A,A_11,A_e68,F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To...,True,1
1,22,anomaly,21,2023-08-12 09:50:00,2023-08-19 10:00:00,Hydraulic group,51888,52892,A,A_21,A_e22,F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To...,True,1
2,72,anomaly,21,2023-10-10 08:40:00,2023-10-17 08:40:00,Gearbox failure,52497,53505,A,A_21,A_e72,F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To...,True,1
3,73,anomaly,0,2023-06-10 11:40:00,2023-06-17 11:40:00,Hydraulic group,52745,53753,A,A_0,A_e73,F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To...,True,1
4,0,anomaly,0,2023-08-06 06:10:00,2023-08-20 06:10:00,Generator bearing failure,52436,54447,A,A_0,A_e0,F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To...,True,1


In [6]:
# ============ CELL 4: LABEL AUDIT ============
n_anom, n_norm = int(EVENTS.is_anomaly.sum()), int((1 - EVENTS.is_anomaly).sum())
print(f"events={len(EVENTS)}  anomaly={n_anom}  normal={n_norm}")
print("v6 expected 45/50 |", "MATCH" if (n_anom, n_norm) == (45, 50) else "MISMATCH -> you may have v<6")

print("\nassets per farm:")
print(EVENTS.groupby("farm").asset_uid.nunique())
print("total unique assets:", EVENTS.asset_uid.nunique(), "(v6 expects 36)")

print("\nevents per asset (min/median/max):")
per = EVENTS.groupby("asset_uid").size()
print(per.describe()[["min", "50%", "max"]].to_dict())

# assets that would leave a split empty
print("\nassets with only one label present:")
lab = EVENTS.groupby("asset_uid").is_anomaly.agg(["sum", "count"])
print(lab[(lab["sum"] == 0) | (lab["sum"] == lab["count"])].shape[0], "of", len(lab))

# free-text fault taxonomy for the discussion section
if "descr" in EVENTS:
    print("\nfault descriptions (anomaly events):")
    for d, k in Counter(EVENTS.loc[EVENTS.is_anomaly == 1, "descr"].astype(str)).most_common(20):
        print(f"  {k:2d}  {d[:90]}")

events=95  anomaly=45  normal=50
v6 expected 45/50 | MATCH

assets per farm:
farm
A     5
B     9
C    22
Name: asset_uid, dtype: int64
total unique assets: 36 (v6 expects 36)

events per asset (min/median/max):
{'min': 1.0, '50%': 2.0, 'max': 6.0}

assets with only one label present:
6 of 36

fault descriptions (anomaly events):
   6  Hydraulic group
   3  high temperature in transformer cell
   2  Gearbox failure
   2  Generator bearing failure
   2  23020 : Axis 3 not ready-to-operate
   1  Transformer failure
   1  Gearbox bearings damaged
   1  Rotor Bearing 2 - Damage
   1  Turbine is stopped due to a main bearing damage
   1  Turbine is in standstill since 01.08 due to rotorbearing damage.
   1  Harting plug Nacelle/HUB damaged + NCR20_HUB: Wiring blade control system
   1  Converter Failure from 17.11 12:30 - 18.11. 13:57, Fuse Filter Supply 
   1  Failure due to Rotorbrake and Hydraulic problemes - Hydraulic pump A disabeld, 2h later tu
   1  10115 : Oil level error, two-pump 

In [7]:
# ============ CELL 5: LEAKAGE AUDIT ============
DESC_CANDS = ["id", "time_stamp", "timestamp", "asset_id", "train_test", "status_type_id"]

def read_desc_only(path):
    sep = sniff_sep(path)
    cols = pd.read_csv(path, sep=sep, nrows=0).columns
    keep = [c for c in cols if c.lower().strip() in DESC_CANDS]
    df = pd.read_csv(path, sep=sep, usecols=keep)
    df.columns = [c.lower().strip() for c in df.columns]
    tcol = "time_stamp" if "time_stamp" in df else "timestamp"
    df[tcol] = pd.to_datetime(df[tcol], errors="coerce")
    return df.rename(columns={tcol: "t"})

spans = []
for _, r in EVENTS.iterrows():
    if not r.file_ok: continue
    d = read_desc_only(r.path)
    tt = d["train_test"].astype(str).str.lower()
    for phase in ("train", "prediction"):
        sel = d[tt.str.startswith(phase[:4])]
        if len(sel) == 0: continue
        spans.append(dict(uid=r.uid, farm=r.farm, asset_uid=r.asset_uid, label=r.label,
                          phase=phase, t0=sel["t"].min(), t1=sel["t"].max(), n=len(sel),
                          n_normal_status=int(sel["status_type_id"].isin(NORMAL_STATUS).sum())
                          if "status_type_id" in sel else -1))
SPANS = pd.DataFrame(spans)
SPANS.to_csv(CACHE / "spans.csv", index=False)
print(SPANS.groupby(["farm", "phase"]).agg(n_events=("uid", "nunique"), rows=("n", "sum")))

# pairwise temporal overlap within the same asset
def overlap_days(a0, a1, b0, b1):
    return max(0.0, (min(a1, b1) - max(a0, b0)).total_seconds() / 86400)

rows = []
for asset, g in SPANS.groupby("asset_uid"):
    g = g.reset_index(drop=True)
    for i in range(len(g)):
        for j in range(i + 1, len(g)):
            a, b = g.loc[i], g.loc[j]
            if a.uid == b.uid: continue
            ov = overlap_days(a.t0, a.t1, b.t0, b.t1)
            if ov > 0:
                rows.append(dict(asset=asset, uid_a=a.uid, phase_a=a.phase,
                                 uid_b=b.uid, phase_b=b.phase, overlap_days=round(ov, 1)))
OVERLAP = pd.DataFrame(rows)
print("\noverlapping span pairs:", len(OVERLAP))
if len(OVERLAP):
    print(OVERLAP.groupby(["phase_a", "phase_b"]).overlap_days.agg(["count", "median", "max"]))
    print("\nWORST OFFENDER: train(one event) vs prediction(another event), same turbine:")
    bad = OVERLAP[(OVERLAP.phase_a != OVERLAP.phase_b)]
    print(bad.sort_values("overlap_days", ascending=False).head(15).to_string(index=False))
OVERLAP.to_csv(CACHE / "overlap_audit.csv", index=False)

                 n_events     rows
farm phase                        
A    prediction        22    50593
     train             22  1146154
B    prediction        15    72128
     train             15   786937
C    prediction        58   158528
     train             58  3028608

overlapping span pairs: 217
                       count  median    max
phase_a    phase_b                         
prediction prediction      5   13.30   21.0
           train          45   15.00   62.0
train      prediction     59   17.00   29.6
           train         108  266.05  364.8

WORST OFFENDER: train(one event) vs prediction(another event), same turbine:
asset uid_a    phase_a uid_b    phase_b  overlap_days
 C_35 C_e67 prediction C_e48      train          62.0
 C_35 C_e67 prediction C_e58      train          62.0
 B_13  B_e7 prediction  B_e2      train          37.0
 A_10 A_e40 prediction  A_e3      train          34.5
 A_10 A_e40 prediction A_e42      train          34.5
 A_10 A_e40 prediction A_

In [8]:
# ============ CELL 6: FEATURE SCHEMA (corrected) ============
STAT_MAP = {"average":"avg", "maximum":"max", "minimum":"min", "std_dev":"std", "stddev":"std", "std":"std"}
UNIT_FIX = {"\ufffdc":"degc", "\ufffd":"deg", "°c":"degc", "°":"deg", "\xb0c":"degc", "\xb0":"deg"}

def read_csv_enc(path, **kw):
    sep = sniff_sep(path)
    for enc in ("utf-8", "cp1252", "latin-1"):
        try:
            df = pd.read_csv(path, sep=sep, encoding=enc, **kw)
            if not df.astype(str).apply(lambda s: s.str.contains("\ufffd", na=False)).any().any():
                return df, enc
        except (UnicodeDecodeError, LookupError):
            continue
    return pd.read_csv(path, sep=sep, encoding="latin-1", **kw), "latin-1"

def fix_unit(u):
    s = str(u).strip()
    for k, v in UNIT_FIX.items(): s = s.replace(k, v)
    s = s.lower().replace("nan", "").replace("none", "").replace("-", "")
    return s.strip()

def load_feature_desc(farm):
    fd, enc = read_csv_enc(ROOT / farm / "feature_description.csv")
    print(f"  {farm}: decoded as {enc}, {len(fd)} base sensors")
    fd.columns = [c.lower().strip() for c in fd.columns]
    rows = []
    for _, r in fd.iterrows():
        stats = [STAT_MAP.get(s.strip().lower()) for s in str(r["statistics_type"]).split(",")]
        stats = [s for s in stats if s]
        for st in stats:
            rows.append(dict(farm=FARM_TAG[farm], base=str(r["sensor_name"]).strip(),
                             name=f"{str(r['sensor_name']).strip()}_{st}", stat_parsed=st,
                             description=str(r["description"]).strip(), unit_n=fix_unit(r["unit"]),
                             is_angle=bool(r["is_angle"]), is_counter=bool(r["is_counter"])))
    return pd.DataFrame(rows)

FEAT = pd.concat([load_feature_desc(f) for f in FARMS], ignore_index=True)
FEAT["desc_n"] = FEAT["description"].str.lower()
FEAT["is_context"] = FEAT["base"].str.contains(r"wind_speed|^power_|reactive_power", case=False, regex=True)

print("\nexpanded columns per farm/stat:")
print(FEAT.groupby(["farm","stat_parsed"]).size().unstack(fill_value=0))

# --- VALIDATE against real headers: this must be near-zero mismatch ---
for farm in FARMS:
    p = sorted((ROOT/farm/"datasets").glob("*.csv"))[0]
    cols = set(c.lower().strip() for c in pd.read_csv(p, sep=sniff_sep(p), nrows=0).columns)
    declared = set(FEAT[FEAT.farm==FARM_TAG[farm]].name.str.lower())
    print(f"{farm}: declared={len(declared)} in-file={len(cols)-5} "
          f"| missing_from_file={len(declared-cols)} | undeclared_in_file={len(cols-declared-set(DESC_CANDS))}")
    if declared - cols: print("   e.g. missing:", sorted(declared-cols)[:5])

QUANTITY_RULES = [
    ("temperature", r"temperat|degc"), ("pressure", r"pressure|bar\b|hpa|\bpa\b"),
    ("speed_rot", r"\brpm\b|rotor speed|generator speed"), ("accel_rot", r"rpm/s|acceleration"),
    ("wind_speed", r"windspeed|wind speed|m/s"), ("power_active", r"active power|\bkw\b|\bmw\b"),
    ("power_reactive", r"reactive|kvar"), ("energy_counter", r"kwh|kvarh|counter|\bh\b$"),
    ("voltage", r"voltage|\bkv\b|\bv\b$"), ("current", r"current|\ba\b$|ampere"),
    ("angle", r"direction|angle|pitch|yaw|azimuth|\bdeg\b"), ("frequency", r"\bhz\b|frequen"),
    ("vibration", r"vibrat|mm/s"), ("flow", r"flow|l/min|m3"), ("humidity", r"humid|%rh"),
    ("percent", r"^%$"),
]
def quantity_of(desc, unit):
    s = f"{desc} {unit}"
    for q, pat in QUANTITY_RULES:
        if re.search(pat, s, re.I): return q
    return "other"

FEAT["quantity"] = [quantity_of(d,u) for d,u in zip(FEAT.desc_n, FEAT.unit_n)]
print("\nquantity x farm (avg channels only):")
print(pd.crosstab(FEAT[FEAT.stat_parsed=="avg"].quantity, FEAT[FEAT.stat_parsed=="avg"].farm))
print("\n'other' share:", round((FEAT[FEAT.stat_parsed=='avg'].quantity=='other').mean(),3))

FEAT["sem_key"] = FEAT.quantity + "|" + FEAT.unit_n + "|" + FEAT.desc_n.str.replace(r"\d+","#",regex=True)
sem = FEAT[FEAT.stat_parsed=="avg"].groupby("sem_key").farm.nunique()
print(f"\nsemantic groups: {len(sem)} | >=2 farms: {(sem>=2).sum()} | all 3: {(sem==3).sum()}")
FEAT.to_csv(CACHE/"feature_schema.csv", index=False, encoding="utf-8")

  Wind Farm A: decoded as cp1252, 54 base sensors
  Wind Farm B: decoded as cp1252, 63 base sensors
  Wind Farm C: decoded as utf-8, 238 base sensors

expanded columns per farm/stat:
stat_parsed  avg  max  min  std
farm                           
A             54    9    9    9
B             63   63   63   63
C            238  238  238  238
Wind Farm A: declared=81 in-file=81 | missing_from_file=8 | undeclared_in_file=8
   e.g. missing: ['sensor_44_avg', 'sensor_45_avg', 'sensor_46_avg', 'sensor_47_avg', 'sensor_48_avg']
Wind Farm B: declared=252 in-file=252 | missing_from_file=0 | undeclared_in_file=0
Wind Farm C: declared=952 in-file=952 | missing_from_file=0 | undeclared_in_file=0

quantity x farm (avg channels only):
farm             A   B   C
quantity                  
angle            4   3  13
current          3   7  47
energy_counter   0   0   1
flow             0   0   2
frequency        1   2   3
other            1   5  12
power_active    13   7  15
pressure         0   0  29

In [9]:
# ============ CELL 7: SENSOR METADATA TOKENS (corrected) ============
COMPS  = ["raw","sin","cos"];               COMP2I = {c:i for i,c in enumerate(COMPS)}
UNITS  = sorted(FEAT.unit_n.unique());      UNIT2I = {u:i for i,u in enumerate(UNITS)}
QUANTS = sorted(FEAT.quantity.unique());    QUANT2I= {q:i for i,q in enumerate(QUANTS)}
STATS  = ["avg","min","max","std"];         STAT2I = {s:i for i,s in enumerate(STATS)}

def desc_hash_vec(text, dim=32):
    v = np.zeros(dim, np.float32)
    for tok in re.findall(r"[a-z]+", str(text).lower()):
        v[int(hashlib.md5(tok.encode()).hexdigest(),16) % dim] += 1.0
    n = np.linalg.norm(v); return v/n if n>0 else v

def build_farm_schema(tag, use_stats=USE_STATS):
    f = FEAT[(FEAT.farm==tag) & (FEAT.stat_parsed.isin(use_stats))].reset_index(drop=True)
    src, comp, rows = [], [], []
    for i, r in f.iterrows():
        for c in (["sin","cos"] if r.is_angle else ["raw"]):
            src.append(r["name"]); comp.append(COMP2I[c]); rows.append(i)
    rows = np.array(rows)
    return dict(
        names   = [f"{s}:{COMPS[c]}" if c else s for s,c in zip(src,comp)],
        src     = src, comp = np.array(comp, np.int64),
        unit    = np.array([UNIT2I[u] for u in f.unit_n], np.int64)[rows],
        quant   = np.array([QUANT2I[q] for q in f.quantity], np.int64)[rows],
        stat    = np.array([STAT2I[s] for s in f.stat_parsed], np.int64)[rows],
        angle   = np.array(f.is_angle.astype(int), np.int64)[rows].astype(np.float32),
        counter = np.array(f.is_counter.astype(int), np.int64)[rows].astype(np.float32),
        context = np.array(f.is_context.astype(int), np.int64)[rows],
        desc    = np.stack([desc_hash_vec(d) for d in f.desc_n])[rows],
        sem_key = [f.sem_key.iloc[i] for i in rows],
    )

SCHEMA = {t: build_farm_schema(t) for t in ["A","B","C"]}
for t,m in SCHEMA.items():
    print(f"Farm {t}: {len(m['names'])} tokens "
          f"(angle-derived: {int((m['comp']>0).sum())}, context: {int(m['context'].sum())})")
np.save(CACHE/"schema.npy", SCHEMA, allow_pickle=True)

Farm A: 58 tokens (angle-derived: 8, context: 6)
Farm B: 66 tokens (angle-derived: 6, context: 6)
Farm C: 250 tokens (angle-derived: 24, context: 11)


In [10]:
# ============ CELL 8: CACHE EVENTS (corrected) ============
NPZ = CACHE/"events"; NPZ.mkdir(exist_ok=True)
DESC_CANDS = ["id","time_stamp","timestamp","asset_id","train_test","status_type_id"]

def cache_event(row, overwrite=False):
    out = NPZ/f"{row.uid}.npz"
    if out.exists() and not overwrite: return out, []
    sep = sniff_sep(row.path)
    cols = pd.read_csv(row.path, sep=sep, nrows=0).columns
    lowmap = {c.lower().strip(): c for c in cols}
    sch = SCHEMA[row.farm]
    need = sorted(set(sch["src"]))
    present = [lowmap[w.lower()] for w in need if w.lower() in lowmap]
    missing = [w for w in need if w.lower() not in lowmap]
    desc = [lowmap[c] for c in DESC_CANDS if c in lowmap]
    df = pd.read_csv(row.path, sep=sep, usecols=desc+present)
    df.columns = [c.lower().strip() for c in df.columns]
    tcol = "time_stamp" if "time_stamp" in df else "timestamp"
    df[tcol] = pd.to_datetime(df[tcol], errors="coerce")
    df = df.sort_values(tcol).reset_index(drop=True)

    raw = {w: pd.to_numeric(df[w.lower()], errors="coerce").to_numpy(np.float32)
           if w.lower() in df else np.full(len(df), np.nan, np.float32) for w in need}
    X = np.empty((len(df), len(sch["src"])), np.float32)
    for k,(s,c) in enumerate(zip(sch["src"], sch["comp"])):
        v = raw[s]
        X[:,k] = v if c==0 else (np.sin(np.deg2rad(v)) if c==1 else np.cos(np.deg2rad(v)))
    np.savez_compressed(out, X=X,
        t=df[tcol].values.astype("datetime64[s]").astype(np.int64),
        train_test=(df["train_test"].astype(str).str.lower().str.startswith("train")).to_numpy(np.int8),
        status=pd.to_numeric(df.get("status_type_id",0), errors="coerce").fillna(-1).to_numpy(np.int16))
    return out, missing

from time import time
t0=time(); miss_log={}
E = EVENTS[EVENTS.file_ok].reset_index(drop=True)
for i,r in E.iterrows():
    _, miss = cache_event(r)
    if miss: miss_log[r.uid]=miss
    print(f"{i+1}/{len(E)} {r.uid}  {time()-t0:.0f}s", end="\r")
print(f"\ndone {time()-t0:.0f}s | events with missing channels: {len(miss_log)}")
if miss_log: print({k:v[:3] for k,v in list(miss_log.items())[:5]})

95/95 C_e60  392s
done 392s | events with missing channels: 22
{'A_e68': ['sensor_44_avg', 'sensor_45_avg', 'sensor_46_avg'], 'A_e22': ['sensor_44_avg', 'sensor_45_avg', 'sensor_46_avg'], 'A_e72': ['sensor_44_avg', 'sensor_45_avg', 'sensor_46_avg'], 'A_e73': ['sensor_44_avg', 'sensor_45_avg', 'sensor_46_avg'], 'A_e0': ['sensor_44_avg', 'sensor_45_avg', 'sensor_46_avg']}


In [11]:
# ============ CELL 6b: FARM A ALIAS REPAIR ============
p = sorted((ROOT/"Wind Farm A"/"datasets").glob("*.csv"))[0]
cols = [c.lower().strip() for c in pd.read_csv(p, sep=sniff_sep(p), nrows=0).columns]
declared = set(FEAT[FEAT.farm=="A"].name.str.lower())
missing  = sorted(declared - set(cols))
extra    = sorted(set(cols) - declared - set(DESC_CANDS))
print("declared but absent:", missing)
print("present but undeclared:", extra)

KEY = re.compile(r"^(?P<pre>[a-z_]+?)_(?P<idx>\d+)_(?P<stat>avg|max|min|std)$")
def keyof(c):
    m = KEY.match(c);  return (int(m.group("idx")), m.group("stat")) if m else None

alias = {}                      # declared_name -> actual_column_in_file
extra_by_key = {keyof(c): c for c in extra if keyof(c)}
for d in missing:
    k = keyof(d)
    if k in extra_by_key: alias[d] = extra_by_key[k]
print(f"\nrepaired by (index,stat) match: {len(alias)}/{len(missing)}")
for k,v in alias.items(): print(f"  {k:22s} -> {v}")
unresolved = [d for d in missing if d not in alias]
print("UNRESOLVED:", unresolved)

declared but absent: ['sensor_44_avg', 'sensor_45_avg', 'sensor_46_avg', 'sensor_47_avg', 'sensor_48_avg', 'sensor_49_avg', 'sensor_50_avg', 'sensor_51_avg']
present but undeclared: ['sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51']

repaired by (index,stat) match: 0/8
UNRESOLVED: ['sensor_44_avg', 'sensor_45_avg', 'sensor_46_avg', 'sensor_47_avg', 'sensor_48_avg', 'sensor_49_avg', 'sensor_50_avg', 'sensor_51_avg']


In [12]:
# ============ CELL 6c: APPLY ALIAS + RECACHE A ============
ALIAS = {"A": alias, "B": {}, "C": {}}
np.save(CACHE/"alias.npy", ALIAS, allow_pickle=True)

# patch the loader: resolve declared -> actual before reading
_orig_cache_event = cache_event
def cache_event(row, overwrite=False):
    out = NPZ/f"{row.uid}.npz"
    if out.exists() and not overwrite: return out, []
    sep = sniff_sep(row.path)
    lowmap = {c.lower().strip(): c for c in pd.read_csv(row.path, sep=sep, nrows=0).columns}
    A = ALIAS.get(row.farm, {})
    sch = SCHEMA[row.farm]; need = sorted(set(sch["src"]))
    resolve_col = lambda w: A.get(w.lower(), w.lower())
    present = [lowmap[resolve_col(w)] for w in need if resolve_col(w) in lowmap]
    missing = [w for w in need if resolve_col(w) not in lowmap]
    desc = [lowmap[c] for c in DESC_CANDS if c in lowmap]
    df = pd.read_csv(row.path, sep=sep, usecols=desc+present)
    df.columns = [c.lower().strip() for c in df.columns]
    tcol = "time_stamp" if "time_stamp" in df else "timestamp"
    df[tcol] = pd.to_datetime(df[tcol], errors="coerce")
    df = df.sort_values(tcol).reset_index(drop=True)
    raw = {w: (pd.to_numeric(df[resolve_col(w)], errors="coerce").to_numpy(np.float32)
               if resolve_col(w) in df else np.full(len(df), np.nan, np.float32)) for w in need}
    X = np.empty((len(df), len(sch["src"])), np.float32)
    for k,(s,c) in enumerate(zip(sch["src"], sch["comp"])):
        v = raw[s]
        X[:,k] = v if c==0 else (np.sin(np.deg2rad(v)) if c==1 else np.cos(np.deg2rad(v)))
    np.savez_compressed(out, X=X, t=df[tcol].values.astype("datetime64[s]").astype(np.int64),
        train_test=(df["train_test"].astype(str).str.lower().str.startswith("train")).to_numpy(np.int8),
        status=pd.to_numeric(df.get("status_type_id",0), errors="coerce").fillna(-1).to_numpy(np.int16))
    return out, missing

for _, r in EVENTS[(EVENTS.farm=="A") & EVENTS.file_ok].iterrows():
    _, m = cache_event(r, overwrite=True)
    if m: print("still missing in", r.uid, m)
print("Farm A re-cached.")

still missing in A_e68 ['sensor_44_avg', 'sensor_45_avg', 'sensor_46_avg', 'sensor_47_avg', 'sensor_48_avg', 'sensor_49_avg', 'sensor_50_avg', 'sensor_51_avg']
still missing in A_e22 ['sensor_44_avg', 'sensor_45_avg', 'sensor_46_avg', 'sensor_47_avg', 'sensor_48_avg', 'sensor_49_avg', 'sensor_50_avg', 'sensor_51_avg']
still missing in A_e72 ['sensor_44_avg', 'sensor_45_avg', 'sensor_46_avg', 'sensor_47_avg', 'sensor_48_avg', 'sensor_49_avg', 'sensor_50_avg', 'sensor_51_avg']
still missing in A_e73 ['sensor_44_avg', 'sensor_45_avg', 'sensor_46_avg', 'sensor_47_avg', 'sensor_48_avg', 'sensor_49_avg', 'sensor_50_avg', 'sensor_51_avg']
still missing in A_e0 ['sensor_44_avg', 'sensor_45_avg', 'sensor_46_avg', 'sensor_47_avg', 'sensor_48_avg', 'sensor_49_avg', 'sensor_50_avg', 'sensor_51_avg']
still missing in A_e26 ['sensor_44_avg', 'sensor_45_avg', 'sensor_46_avg', 'sensor_47_avg', 'sensor_48_avg', 'sensor_49_avg', 'sensor_50_avg', 'sensor_51_avg']
still missing in A_e40 ['sensor_44_avg', 

In [13]:
# ============ CELL 6d: SOFT SEMANTIC ALIGNMENT ============
FEAT["coarse_key"] = FEAT.quantity + "|" + FEAT.unit_n          # physics, not wording
avg = FEAT[FEAT.stat_parsed=="avg"]
ck = avg.groupby("coarse_key").farm.nunique()
print(f"coarse groups: {len(ck)} | >=2 farms: {(ck>=2).sum()} | all 3 farms: {(ck==3).sum()}")
print("\nchannels living in an all-3-farm group:")
shared = set(ck[ck==3].index)
print(avg.assign(sh=avg.coarse_key.isin(shared)).groupby("farm").sh.agg(["sum","count","mean"]))

# richer text embedding: word tokens + char trigrams, 64-dim (32 was collision-heavy)
def desc_vec(text, dim=64):
    s = re.sub(r"[^a-z0-9 ]", " ", str(text).lower())
    toks = re.findall(r"[a-z]+", s) + [s[i:i+3] for i in range(max(len(s)-2,0))]
    v = np.zeros(dim, np.float32)
    for t in toks: v[int(hashlib.md5(t.encode()).hexdigest(),16) % dim] += 1.0
    n = np.linalg.norm(v); return v/n if n>0 else v

# sanity: does the embedding pair semantically-equivalent sensors across farms?
def nn_pairs(f1, f2, topn=12):
    a = avg[avg.farm==f1]; b = avg[avg.farm==f2]
    Va = np.stack([desc_vec(d) for d in a.desc_n]); Vb = np.stack([desc_vec(d) for d in b.desc_n])
    S = Va @ Vb.T
    same = (a.quantity.values[:,None] == b.quantity.values[None,:])
    S = S * same                                   # only compare within a physical quantity
    best = S.argmax(1); score = S.max(1)
    o = np.argsort(-score)[:topn]
    return pd.DataFrame({f"{f1}_desc": a.description.values[o],
                         f"{f2}_desc": b.description.values[o[0:0].tolist() or slice(None)][:0].tolist()
                                        or b.description.values[best[o]],
                         "cos": score[o].round(3), "quantity": a.quantity.values[o]})

print("\nA -> C nearest semantic matches:"); print(nn_pairs("A","C").to_string(index=False))
print("\nA -> B nearest semantic matches:"); print(nn_pairs("A","B").to_string(index=False))

coarse groups: 36 | >=2 farms: 10 | all 3 farms: 6

channels living in an all-3-farm group:
      sum  count      mean
farm                      
A      14     54  0.259259
B      19     63  0.301587
C      95    238  0.399160

A -> C nearest semantic matches:
                                                 A_desc                                    C_desc   cos     quantity
                                    Ambient temperature                       Ambient temperature 1.000  temperature
                                Wind relative direction                 Relative wind direction 2 0.894        angle
                                    Nacelle temperature               Nacelle outside temperature 0.864  temperature
                                    Grid reactive power                    Reactive power HV grid 0.852 power_active
                                     Total active power             Active power aeration motor A 0.829 power_active
                 Possible grid induct

In [14]:
# ============ CELL 6b: FARM A ALIAS REPAIR (v2) ============
p = sorted((ROOT/"Wind Farm A"/"datasets").glob("*.csv"))[0]
cols = [c.lower().strip() for c in pd.read_csv(p, sep=sniff_sep(p), nrows=0).columns]
declared = set(FEAT[FEAT.farm=="A"].name.str.lower())
missing = sorted(declared - set(cols))
extra   = sorted(set(cols) - declared - set(DESC_CANDS))

alias = {}
for d in missing:                                  # sensor_44_avg -> sensor_44
    base = re.sub(r"_(avg|max|min|std)$", "", d)
    if base in extra: alias[d] = base
print(f"repaired: {len(alias)}/{len(missing)}")
print("UNRESOLVED:", [d for d in missing if d not in alias])

# what are they? (justifies the export quirk in the paper)
print(FEAT[FEAT.name.str.lower().isin(alias)][["name","description","unit_n","is_counter"]].to_string(index=False))

repaired: 8/8
UNRESOLVED: []
         name                                   description  unit_n  is_counter
sensor_44_avg         Active power - generator disconnected      wh       False
sensor_45_avg   Active power - generator connected in delta      wh       False
sensor_46_avg    Active power - generator connected in star      wh       False
sensor_47_avg       Reactive power - generator disconnected    varh       False
sensor_48_avg Reactive power - generator connected in delta    varh       False
sensor_49_avg  Reactive power - generator connected in star    varh       False
sensor_50_avg                            Total active power      wh       False
sensor_51_avg                          Total reactive power    varh       False
sensor_44_avg     Transformer L2 (undervoltage) temperature    ï¿½c       False
sensor_45_avg      Transformer L3 (mid-voltage) temperature    ï¿½c       False
sensor_46_avg     Transformer L3 (undervoltage) temperature    ï¿½c       False
sensor_47_a

In [15]:
# ============ CELL 6e: QUANTITY RULE FIX + REBUILD ============
QUANTITY_RULES = [
    ("power_reactive", r"reactive|kvar"),                      # MUST precede power_active
    ("energy_counter", r"kwh|kvarh|counter|\bh\b$"),
    ("temperature", r"temperat|degc"), ("pressure", r"pressure|bar\b|hpa|\bpa\b"),
    ("speed_rot", r"\brpm\b|rotor speed|generator speed"), ("accel_rot", r"rpm/s|acceleration"),
    ("wind_speed", r"windspeed|wind speed|m/s"),
    ("power_active", r"(?<!re)active power|\bkw\b|\bmw\b"),
    ("voltage", r"voltage|\bkv\b|\bv\b$"), ("current", r"current|\ba\b$|ampere"),
    ("angle", r"direction|angle|pitch|yaw|azimuth|\bdeg\b"), ("frequency", r"\bhz\b|frequen"),
    ("vibration", r"vibrat|mm/s"), ("flow", r"flow|l/min|m3"), ("humidity", r"humid|%rh"),
    ("percent", r"^%$"),
]
def quantity_of(desc, unit):
    s = f"{desc} {unit}"
    for q, pat in QUANTITY_RULES:
        if re.search(pat, s, re.I): return q
    return "other"

FEAT["quantity"] = [quantity_of(d,u) for d,u in zip(FEAT.desc_n, FEAT.unit_n)]
FEAT["coarse_key"] = FEAT.quantity + "|" + FEAT.unit_n
FEAT["sem_key"] = FEAT.quantity + "|" + FEAT.unit_n + "|" + FEAT.desc_n.str.replace(r"\d+","#",regex=True)
avg = FEAT[FEAT.stat_parsed=="avg"]
print(pd.crosstab(avg.quantity, avg.farm))
ck = avg.groupby("coarse_key").farm.nunique()
print(f"\ncoarse groups: {len(ck)} | >=2 farms: {(ck>=2).sum()} | all 3: {(ck==3).sum()}")

def desc_vec(text, dim=64):
    s = re.sub(r"[^a-z0-9 ]", " ", str(text).lower())
    toks = re.findall(r"[a-z]+", s) + [s[i:i+3] for i in range(max(len(s)-2,0))]
    v = np.zeros(dim, np.float32)
    for t in toks: v[int(hashlib.md5(t.encode()).hexdigest(),16) % dim] += 1.0
    n = np.linalg.norm(v); return v/n if n>0 else v

UNITS  = sorted(FEAT.unit_n.unique());   UNIT2I  = {u:i for i,u in enumerate(UNITS)}
QUANTS = sorted(FEAT.quantity.unique()); QUANT2I = {q:i for i,q in enumerate(QUANTS)}

def build_farm_schema(tag, use_stats=USE_STATS, dim=64):
    f = FEAT[(FEAT.farm==tag) & (FEAT.stat_parsed.isin(use_stats))].reset_index(drop=True)
    src, comp, rows = [], [], []
    for i, r in f.iterrows():
        for c in (["sin","cos"] if r.is_angle else ["raw"]):
            src.append(r["name"]); comp.append(COMP2I[c]); rows.append(i)
    rows = np.array(rows)
    return dict(names=[f"{s}:{COMPS[c]}" if c else s for s,c in zip(src,comp)],
        src=src, comp=np.array(comp,np.int64),
        unit=np.array([UNIT2I[u] for u in f.unit_n],np.int64)[rows],
        quant=np.array([QUANT2I[q] for q in f.quantity],np.int64)[rows],
        stat=np.array([STAT2I[s] for s in f.stat_parsed],np.int64)[rows],
        angle=np.array(f.is_angle.astype(int))[rows].astype(np.float32),
        counter=np.array(f.is_counter.astype(int))[rows].astype(np.float32),
        context=np.array(f.is_context.astype(int),np.int64)[rows],
        desc=np.stack([desc_vec(d,dim) for d in f.desc_n])[rows],
        sem_key=[f.sem_key.iloc[i] for i in rows])

SCHEMA = {t: build_farm_schema(t) for t in ["A","B","C"]}
for t,m in SCHEMA.items(): print(f"Farm {t}: {len(m['names'])} tokens, desc dim {m['desc'].shape[1]}")
np.save(CACHE/"schema.npy", SCHEMA, allow_pickle=True)
np.save(CACHE/"alias.npy", {"A":alias,"B":{},"C":{}}, allow_pickle=True)

farm             A   B   C
quantity                  
angle            4   3  13
current          3   7  47
energy_counter   0   2   1
flow             0   0   2
frequency        1   2   3
other            1   5  12
power_active     6   2  10
power_reactive   7   3   5
pressure         0   0  29
speed_rot        2   3   8
temperature     25  25  72
vibration        0   3   0
voltage          3   4  29
wind_speed       2   4   7

coarse groups: 36 | >=2 farms: 10 | all 3: 6
Farm A: 58 tokens, desc dim 64
Farm B: 66 tokens, desc dim 64
Farm C: 250 tokens, desc dim 64


In [17]:
# ============ CELL 6f: UNIT NORMALISATION (v2) ============
UNIT_FIX = {"ï¿½c":"degc", "ï¿½":"deg", "\ufffdc":"degc", "\ufffd":"deg",
            "°c":"degc", "°":"deg", "\xb0c":"degc", "\xb0":"deg", "celsius":"degc"}
def fix_unit(u):
    s = str(u).strip().lower()
    for k,v in UNIT_FIX.items(): s = s.replace(k, v)
    return re.sub(r"\s+", "", s.replace("nan","").replace("none","").replace("-",""))

FEAT["unit_n"] = [fix_unit(u) for u in FEAT["unit"]] if "unit" in FEAT else FEAT["unit_n"].map(fix_unit)
print("unit vocabulary:", sorted(FEAT.unit_n.unique()))

# energy registers are NOT power: split them out before the power rules
QUANTITY_RULES = [
    ("energy_active",   r"^(wh|kwh|mwh)$"),
    ("energy_reactive", r"^(varh|kvarh|mvarh)$"),
    ("power_reactive",  r"reactive|kvar"),
    ("temperature", r"temperat|degc"), ("pressure", r"pressure|bar\b|hpa|\bpa\b"),
    ("speed_rot", r"\brpm\b|rotor speed|generator speed"), ("accel_rot", r"rpm/s|acceleration"),
    ("wind_speed", r"windspeed|wind speed|m/s"),
    ("power_active", r"(?<!re)active power|\bkw\b|\bmw\b"),
    ("voltage", r"voltage|\bkv\b|\bv\b$"), ("current", r"current|\ba\b$|ampere"),
    ("angle", r"direction|angle|pitch|yaw|azimuth|\bdeg\b"), ("frequency", r"\bhz\b|frequen"),
    ("vibration", r"vibrat|mm/s"), ("level", r"level|fluid"), ("flow", r"flow|l/min|m3"),
    ("humidity", r"humid|%rh"), ("percent", r"^%$"),
]
def quantity_of(desc, unit):
    for q, pat in QUANTITY_RULES:                      # unit-only rules first
        if q.startswith("energy_") and re.search(pat, str(unit), re.I): return q
    for q, pat in QUANTITY_RULES:
        if not q.startswith("energy_") and re.search(pat, f"{desc} {unit}", re.I): return q
    return "other"

FEAT["quantity"]   = [quantity_of(d,u) for d,u in zip(FEAT.desc_n, FEAT.unit_n)]
FEAT["coarse_key"] = FEAT.quantity + "|" + FEAT.unit_n
FEAT["sem_key"]    = FEAT.quantity + "|" + FEAT.unit_n + "|" + FEAT.desc_n.str.replace(r"\d+","#",regex=True)
avg = FEAT[FEAT.stat_parsed=="avg"]
print(pd.crosstab(avg.quantity, avg.farm))
ck = avg.groupby("coarse_key").farm.nunique()
print(f"coarse groups: {len(ck)} | >=2 farms: {(ck>=2).sum()} | all 3: {(ck==3).sum()}")

UNITS  = sorted(FEAT.unit_n.unique());   UNIT2I  = {u:i for i,u in enumerate(UNITS)}
QUANTS = sorted(FEAT.quantity.unique()); QUANT2I = {q:i for i,q in enumerate(QUANTS)}
SCHEMA = {t: build_farm_schema(t) for t in ["A","B","C"]}
np.save(CACHE/"schema.npy", SCHEMA, allow_pickle=True)
for t,m in SCHEMA.items(): print(f"Farm {t}: {len(m['names'])} tokens")

unit vocabulary: ['', '%', '1/min', 'a', 'bar', 'deg', 'degc', 'dl', 'hpa', 'hz', 'knm', 'kva', 'kvar', 'kvarh', 'kw', 'kwh', 'l', 'l/h', 'l/min', 'm/s', 'm/s^2', 'mg', 'mhz', 'nm', 'pa', 'rad/s', 'rpm', 'rpm/s', 'v', 'varh', 'wh']
farm              A   B   C
quantity                   
angle             4   3  13
current           3   7  47
energy_active     4   2   0
energy_reactive   4   2   0
flow              0   0   3
frequency         1   2   3
level             0   0   4
other             1   5   3
power_active      2   2  10
power_reactive    3   1   5
pressure          0   0  29
speed_rot         2   3   8
temperature      25  25  77
vibration         0   3   0
voltage           3   4  29
wind_speed        2   4   7
coarse groups: 34 | >=2 farms: 10 | all 3: 8
Farm A: 58 tokens
Farm B: 66 tokens
Farm C: 250 tokens


In [22]:
# ============ CELL 9: LOADERS + NORM STATS ============
def load_cached_raw(uid):
    z = np.load(NPZ/f"{uid}.npz")
    return z["X"], z["t"], z["train_test"].astype(bool), z["status"]

load_cached = load_cached_raw          # 8c rebinds this to the counter-aware version

def healthy_mask(train_test, status):
    return train_test & np.isin(status, list(NORMAL_STATUS))

def fit_norm_stats(uids, farm):
    S = len(SCHEMA[farm]["names"])
    n=np.zeros(S); s1=np.zeros(S); s2=np.zeros(S); nan=np.zeros(S)
    for uid in uids:
        X,t,tt,st = load_cached(uid)
        Xi = X[healthy_mask(tt,st)]
        if len(Xi)==0: continue
        fin = np.isfinite(Xi)
        nan += (~fin).sum(0); n += fin.sum(0)
        s1 += np.where(fin, Xi, 0).sum(0)
        s2 += np.where(fin, Xi**2, 0).sum(0)
    mu = np.where(n>0, s1/np.maximum(n,1), 0.0)
    sd = np.sqrt(np.maximum(np.where(n>0, s2/np.maximum(n,1)-mu**2, 1.0), 1e-8))
    return dict(mu=mu.astype(np.float32), sd=sd.astype(np.float32),
                keep=(n>500)&(sd>1e-6), nan_rate=(nan/np.maximum(n+nan,1)).astype(np.float32))

def zscore(X, st): return (X - st["mu"]) / st["sd"]

stA = fit_norm_stats(EVENTS[EVENTS.farm=="A"].uid.tolist(), "A")
print("Farm A kept:", int(stA["keep"].sum()), "/", len(stA["keep"]),
      "| median NaN rate:", round(float(np.median(stA["nan_rate"])),4))

Farm A kept: 50 / 58 | median NaN rate: 0.0


In [23]:
# ============ CELL 8b: MONOTONIC COUNTER AUDIT ============
def monotone_frac(x):
    d = np.diff(x[np.isfinite(x)])
    return float((d >= -1e-9).mean()) if len(d) > 100 else np.nan

COUNTER_IDX = {}
for tag in ["A","B","C"]:
    uids = EVENTS[(EVENTS.farm==tag) & EVENTS.file_ok].uid.tolist()[:6]
    S = len(SCHEMA[tag]["names"]); acc = np.full((len(uids), S), np.nan)
    for i,u in enumerate(uids):
        X,t,tt,st = load_cached(u); H = healthy_mask(tt,st)
        for j in range(S): acc[i,j] = monotone_frac(X[H,j])
    mf = np.nanmean(acc,0)
    flag = SCHEMA[tag]["counter"].astype(bool)
    detected = np.where(mf > 0.98)[0]
    COUNTER_IDX[tag] = detected
    print(f"Farm {tag}: metadata is_counter={int(flag.sum())} | detected monotonic={len(detected)}")
    for j in detected:
        print(f"   {SCHEMA[tag]['names'][j]:22s} mono={mf[j]:.3f} flag={bool(flag[j])} "
              f"q={QUANTS[SCHEMA[tag]['quant'][j]]}")
np.save(CACHE/"counters.npy", COUNTER_IDX, allow_pickle=True)

Farm A: metadata is_counter=0 | detected monotonic=1
   sensor_26_avg          mono=0.986 flag=False q=frequency
Farm B: metadata is_counter=4 | detected monotonic=1
   sensor_9_avg           mono=0.999 flag=False q=frequency
Farm C: metadata is_counter=0 | detected monotonic=7
   sensor_16_avg:sin      mono=0.991 flag=False q=angle
   sensor_16_avg:cos      mono=0.991 flag=False q=angle
   sensor_22_avg          mono=0.995 flag=False q=current
   sensor_97_avg          mono=0.983 flag=False q=pressure
   sensor_98_avg          mono=0.985 flag=False q=pressure
   sensor_99_avg          mono=0.981 flag=False q=pressure
   sensor_213_avg         mono=0.997 flag=False q=voltage


In [24]:
# ============ CELL 8c: COUNTER-AWARE LOADER ============
_raw_load = load_cached
def load_cached(uid):
    X, t, tt, st = _raw_load(uid)
    tag = uid.split("_")[0]
    idx = COUNTER_IDX.get(tag, np.array([], int))
    if len(idx):
        D = np.diff(X[:, idx], axis=0, prepend=np.nan)
        D[D < 0] = np.nan                       # register rollover / reset
        X = X.copy(); X[:, idx] = D             # -> per-interval energy, stationary
    return X, t, tt, st
print("counter channels differenced:", {k: len(v) for k,v in COUNTER_IDX.items()})

counter channels differenced: {'A': 1, 'B': 1, 'C': 7}


In [25]:
ALIAS = {"A": alias, "B": {}, "C": {}}
np.save(CACHE/"alias.npy", ALIAS, allow_pickle=True)

load_cached = load_cached_raw            # undo 8c rebinding before re-caching
for _, r in EVENTS[(EVENTS.farm=="A") & EVENTS.file_ok].iterrows():
    _, m = cache_event(r, overwrite=True)
    if m: print("STILL MISSING", r.uid, m)

stA = fit_norm_stats(EVENTS[EVENTS.farm=="A"].uid.tolist(), "A")
print("Farm A kept:", int(stA["keep"].sum()), "/", len(stA["keep"]), "  <-- expect 58/58")

Farm A kept: 58 / 58   <-- expect 58/58


In [26]:
# ============ CELL 8b: COUNTER / DEAD-CHANNEL AUDIT (v2) ============
def channel_profile(x):
    x = x[np.isfinite(x)]
    if len(x) < 500: return dict(kind="sparse", drift=np.nan, pos=np.nan, rng=np.nan)
    d = np.diff(x); tv = np.abs(d).sum()
    rng = float(x.max() - x.min())
    if tv < 1e-9 or rng < 1e-9: return dict(kind="constant", drift=np.nan, pos=0.0, rng=rng)
    drift = float(abs(x[-1] - x[0]) / tv)          # 1.0 = pure monotone ramp, ~0 = stationary
    pos   = float((d > 0).mean())
    kind = "counter" if (drift > 0.9 and pos > 0.5) else "normal"
    return dict(kind=kind, drift=drift, pos=pos, rng=rng)

COUNTER_IDX, DEAD_IDX = {}, {}
for tag in ["A","B","C"]:
    uids = EVENTS[(EVENTS.farm==tag) & EVENTS.file_ok].uid.tolist()[:6]
    S = len(SCHEMA[tag]["names"])
    votes = defaultdict(list)
    for u in uids:
        X,t,tt,st = load_cached_raw(u); H = healthy_mask(tt,st)
        for j in range(S): votes[j].append(channel_profile(X[H,j]))
    cnt, dead = [], []
    for j,ps in votes.items():
        kinds = Counter(p["kind"] for p in ps)
        if kinds["counter"] >= max(2, len(ps)//2): cnt.append(j)
        elif kinds["constant"] + kinds["sparse"] >= len(ps): dead.append(j)
    COUNTER_IDX[tag], DEAD_IDX[tag] = np.array(cnt,int), np.array(dead,int)
    print(f"\nFarm {tag}: counters={len(cnt)}  dead/constant={len(dead)}  "
          f"(metadata is_counter={int(SCHEMA[tag]['counter'].sum())})")
    for j in cnt:
        d = np.nanmean([p['drift'] for p in votes[j] if np.isfinite(p['drift'])])
        print(f"   COUNTER {SCHEMA[tag]['names'][j]:24s} drift={d:.3f} q={QUANTS[SCHEMA[tag]['quant'][j]]}")
    for j in dead[:8]: print(f"   dead    {SCHEMA[tag]['names'][j]}")

np.save(CACHE/"counters.npy", {"counter":COUNTER_IDX, "dead":DEAD_IDX}, allow_pickle=True)


Farm A: counters=0  dead/constant=2  (metadata is_counter=0)
   dead    sensor_46_avg
   dead    sensor_49_avg

Farm B: counters=0  dead/constant=0  (metadata is_counter=4)

Farm C: counters=0  dead/constant=0  (metadata is_counter=0)


In [27]:
# ============ CELL 8c: COUNTER-AWARE LOADER (v2) ============
def load_cached(uid):
    X, t, tt, st = load_cached_raw(uid)
    tag = uid.split("_")[0]
    ci, di = COUNTER_IDX.get(tag, np.array([],int)), DEAD_IDX.get(tag, np.array([],int))
    X = X.copy()
    if len(ci):
        D = np.diff(X[:, ci], axis=0, prepend=np.nan); D[D < 0] = np.nan   # rollover
        X[:, ci] = D
    if len(di): X[:, di] = np.nan            # fit_norm_stats' keep-filter will drop these
    return X, t, tt, st

for tag in ["A","B","C"]:
    st = fit_norm_stats(EVENTS[(EVENTS.farm==tag) & EVENTS.file_ok].uid.tolist(), tag)
    globals()[f"stats_{tag}"] = st
    print(f"Farm {tag}: kept {int(st['keep'].sum())}/{len(st['keep'])} channels")

Farm A: kept 56/58 channels
Farm B: kept 66/66 channels
Farm C: kept 250/250 channels


In [28]:
for tag, names in [("A", ["sensor_44_avg","sensor_50_avg"]), ("B", ["sensor_0_avg","sensor_1_avg"])]:
    u = EVENTS[(EVENTS.farm==tag) & EVENTS.file_ok].uid.iloc[0]
    X,t,tt,st = load_cached_raw(u); H = healthy_mask(tt,st)
    for n in names:
        j = SCHEMA[tag]["names"].index(n)
        x = X[H,j]; x = x[np.isfinite(x)]
        p = channel_profile(X[H,j])
        print(f"{tag} {n:16s} drift={p['drift']:.3f} pos={p['pos']:.2f} "
              f"min={x.min():.1f} max={x.max():.1f} mean={x.mean():.1f}")

A sensor_44_avg    drift=0.000 pos=0.21 min=-4662.0 max=13.0 mean=-390.0
A sensor_50_avg    drift=0.000 pos=0.51 min=-4662.0 max=333853.0 mean=93918.1
B sensor_0_avg     drift=0.000 pos=0.06 min=0.0 max=379.0 mean=2.6
B sensor_1_avg     drift=0.000 pos=0.05 min=0.0 max=11.0 mean=0.3


In [29]:
# ============ CELL 11: DATASET (memory-safe) ============
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from functools import lru_cache
DEV = "cuda" if torch.cuda.is_available() else "cpu"; print(DEV)

class WindowSet(Dataset):
    """Healthy windows from one farm. Events loaded lazily, LRU-capped."""
    def __init__(self, uids, farm, stats, window=WINDOW, stride=6, healthy_only=True, cache_events=8):
        self.farm, self.st, self.L, self.healthy_only = farm, stats, window, healthy_only
        self.keep = np.where(stats["keep"])[0]
        self._get = lru_cache(maxsize=cache_events)(self._load)
        self.items = []
        for uid in uids:
            X, t, tt, s = load_cached(uid)
            m = healthy_mask(tt, s) if healthy_only else np.ones(len(X), bool)
            ok = np.isfinite(X[:, self.keep]).mean(1) > 0.5
            m = m & ok
            run = np.concatenate([[0], np.cumsum(m)])
            for i in range(window, len(X), stride):
                if run[i+1] - run[i-window] == window + 1: self.items.append((uid, i))
        ctx = SCHEMA[farm]["context"][self.keep].astype(bool)
        self.ctx_idx, self.score_idx = np.where(ctx)[0], np.where(~ctx)[0]
        print(f"  WindowSet: {len(self.items)} windows from {len(uids)} events, "
              f"{len(self.keep)} channels")

    def _load(self, uid):
        X, t, tt, s = load_cached(uid)
        return zscore(X, self.st)[:, self.keep].astype(np.float32)

    def __len__(self): return len(self.items)

    def __getitem__(self, k):
        uid, i = self.items[k]
        Xz = self._get(uid)
        w, y = Xz[i-self.L:i].T, Xz[i]
        return (torch.from_numpy(np.nan_to_num(w)), torch.from_numpy(np.isfinite(w).astype(np.float32)),
                torch.from_numpy(np.nan_to_num(y)), torch.from_numpy(np.isfinite(y).astype(np.float32)))

def meta_tensors(farm, keep_idx):
    m = SCHEMA[farm]
    return dict(unit=torch.as_tensor(m["unit"][keep_idx]), quant=torch.as_tensor(m["quant"][keep_idx]),
                stat=torch.as_tensor(m["stat"][keep_idx]), angle=torch.as_tensor(m["angle"][keep_idx]).float(),
                counter=torch.as_tensor(m["counter"][keep_idx]).float(),
                desc=torch.as_tensor(m["desc"][keep_idx]).float())

cuda


In [31]:
# ============ CELL 10: PROTOCOLS ============
def protocol_splits(protocol, farm=None):
    E = EVENTS[EVENTS.file_ok]
    if protocol == "cross_asset":
        for f in (["A","B","C"] if farm is None else [farm]):
            Ef = E[E.farm == f]
            for a in sorted(Ef.asset_uid.unique()):
                tgt, src = Ef[Ef.asset_uid == a], Ef[Ef.asset_uid != a]
                if len(src) == 0 or len(tgt) == 0: continue
                yield dict(name=f"crossasset_{a}", source=src.uid.tolist(), target=tgt.uid.tolist(),
                           target_farm=f, target_asset=a, cal_uids=[])
    elif protocol == "cross_farm":
        for f in ["A","B","C"]:
            yield dict(name=f"crossfarm_{f}", source=E[E.farm != f].uid.tolist(),
                       target=E[E.farm == f].uid.tolist(), target_farm=f, target_asset=None, cal_uids=[])
    elif protocol == "within_asset":
        for f in ["A","B","C"]:
            for a in sorted(E[E.farm == f].asset_uid.unique()):
                g = E[E.asset_uid == a]
                yield dict(name=f"within_{a}", source=g.uid.tolist(), target=g.uid.tolist(),
                           target_farm=f, target_asset=a, cal_uids=[])
    else: raise ValueError(protocol)

SPLITS = {p: list(protocol_splits(p)) for p in ["within_asset","cross_asset","cross_farm"]}
for k,v in SPLITS.items(): print(k, len(v), "folds")
print("\nFarm A cross-asset folds:")
for f in SPLITS["cross_asset"]:
    if f["target_farm"]=="A":
        print(f"  {f['name']:20s} src={len(f['source'])} tgt={len(f['target'])}")

within_asset 36 folds
cross_asset 36 folds
cross_farm 3 folds

Farm A cross-asset folds:
  crossasset_A_0       src=17 tgt=5
  crossasset_A_10      src=17 tgt=5
  crossasset_A_11      src=18 tgt=4
  crossasset_A_13      src=18 tgt=4
  crossasset_A_21      src=18 tgt=4


In [32]:
for n in ["SCHEMA","SPLITS","WindowSet","SAGEWT","train_sagewt","score_event",
          "calibrate","conformal_p","e_process","evaluate_event","event_metrics","run_fold"]:
    print(f"{n:16s}", "OK" if n in globals() else "MISSING -> re-run its cell")

SCHEMA           OK
SPLITS           OK
WindowSet        OK
SAGEWT           MISSING -> re-run its cell
train_sagewt     MISSING -> re-run its cell
score_event      MISSING -> re-run its cell
calibrate        MISSING -> re-run its cell
conformal_p      MISSING -> re-run its cell
e_process        MISSING -> re-run its cell
evaluate_event   MISSING -> re-run its cell
event_metrics    MISSING -> re-run its cell
run_fold         MISSING -> re-run its cell


In [33]:
fold = [f for f in SPLITS["cross_asset"] if f["target_farm"]=="A"][0]
m, r = run_fold(fold, epochs=4)
print({k: (round(v,3) if isinstance(v,float) else v) for k,v in m.items()})

NameError: name 'run_fold' is not defined

In [34]:
# ============ CELL 12: SAGE-WT MODEL ============
class TemporalEncoder(nn.Module):
    def __init__(self, d=128):
        super().__init__()
        ch, layers, inc = 64, [], 2
        for dil in (1,2,4,8,16):
            layers += [nn.Conv1d(inc, ch, 3, padding=dil, dilation=dil), nn.GELU(), nn.BatchNorm1d(ch)]
            inc = ch
        self.net = nn.Sequential(*layers); self.proj = nn.Linear(ch*2, d)
    def forward(self, x):
        h = self.net(x)
        return self.proj(torch.cat([h.mean(-1), h.max(-1).values], -1))

class MetaEncoder(nn.Module):
    def __init__(self, d=128, desc_dim=64):
        super().__init__()
        self.u = nn.Embedding(len(UNITS), 32); self.q = nn.Embedding(len(QUANTS), 32)
        self.s = nn.Embedding(len(STATS), 8)
        self.mlp = nn.Sequential(nn.Linear(32+32+8+2+desc_dim, d), nn.GELU(), nn.Linear(d, d))
    def forward(self, m):
        z = torch.cat([self.u(m["unit"]), self.q(m["quant"]), self.s(m["stat"]),
                       m["angle"][:,None], m["counter"][:,None], m["desc"]], -1)
        return self.mlp(z)

class SAGEWT(nn.Module):
    def __init__(self, d=128, nhead=4, nlayers=4, desc_dim=64):
        super().__init__()
        self.temp, self.meta = TemporalEncoder(d), MetaEncoder(d, desc_dim)
        enc = nn.TransformerEncoderLayer(d, nhead, 4*d, 0.1, batch_first=True, norm_first=True)
        self.set = nn.TransformerEncoder(enc, nlayers)          # no positional encoding
        self.mask_tok = nn.Parameter(torch.randn(d)*.02)
        self.film = nn.Sequential(nn.Linear(8, d), nn.GELU(), nn.Linear(d, 2*d))
        self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, 2))
    def forward(self, w, obs, meta, ctx_vec, sensor_mask=None):
        B, S, L = w.shape
        x = torch.stack([w, obs], 2).reshape(B*S, 2, L)
        h = self.temp(x).reshape(B, S, -1) + self.meta(meta)[None]
        if sensor_mask is not None:
            h = torch.where(sensor_mask[...,None].bool(), self.mask_tok.expand_as(h), h)
        g, b = self.film(ctx_vec).chunk(2, -1)
        h = self.set(h*(1+g[:,None]) + b[:,None])
        o = self.head(h)
        return o[...,0], o[...,1].clamp(-6, 4)

In [35]:
# ============ CELL 13: MASKING + LOSS ============
def make_masks(B, S, L, p_sensor=0.25, p_time=0.25, dev="cpu"):
    sm = (torch.rand(B, S, device=dev) < p_sensor).float()
    tm = torch.zeros(B, 1, L, device=dev)
    span = max(1, int(0.15*L))
    for i, s0 in enumerate(torch.randint(0, L-span, (B,), device=dev)):
        if torch.rand(1).item() < p_time: tm[i,0,s0:s0+span] = 1
    return sm, tm

def gnll(mu, logsig, y, obs, weight=None):
    l = 0.5*(torch.exp(-2*logsig)*(y-mu)**2 + 2*logsig) * obs
    if weight is not None: l = l*weight
    return l.sum()/obs.sum().clamp(min=1)

def context_vector(w, ctx_idx, dev):
    c = w[:, ctx_idx, -1]
    c = c[:,:6] if c.shape[1] >= 6 else F.pad(c, (0, 6-c.shape[1]))
    return torch.cat([c, c.mean(1,keepdim=True), c.std(1,keepdim=True)], -1).to(dev)

In [36]:
# ============ CELL 14: TRAIN ============
def train_sagewt(source_uids, farm, stats, epochs=8, bs=32, lr=3e-4, p_sensor=0.25,
                 p_time=0.25, kill_meta=False, kill_film=False, verbose=True):
    ds = WindowSet(source_uids, farm, stats)
    dl = DataLoader(ds, batch_size=bs, shuffle=True, drop_last=True, num_workers=0)
    meta = {k: v.to(DEV) for k,v in meta_tensors(farm, np.where(stats["keep"])[0]).items()}
    model = SAGEWT().to(DEV)
    if kill_meta:
        for p in model.meta.parameters(): p.requires_grad_(False); p.data.zero_()
    if kill_film:
        for p in model.film.parameters(): p.requires_grad_(False); p.data.zero_()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, lr, epochs*max(len(dl),1))
    for ep in range(epochs):
        model.train(); tot=nb=0
        for w, ow, y, oy in dl:
            w,ow,y,oy = [t.to(DEV) for t in (w,ow,y,oy)]
            B,S,L = w.shape
            sm, tm = make_masks(B,S,L,p_sensor,p_time,DEV)
            ctx = context_vector(w, ds.ctx_idx, DEV)
            mu, ls = model(w*(1-tm), ow*(1-tm), meta, ctx, sensor_mask=sm)
            loss = gnll(mu, ls, y, oy, 1.0+2.0*sm)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sch.step()
            tot += loss.item(); nb += 1
        if verbose: print(f"  ep{ep+1}/{epochs} nll={tot/max(nb,1):.4f}")
    return model, ds, meta

In [37]:
# ============ CELL 15: SCORE AN EVENT ============
@torch.no_grad()
def score_event(model, uid, farm, stats, ds_ref, meta, batch=256, stride=1):
    model.eval()
    keep = np.where(stats["keep"])[0]
    X, t, tt, st = load_cached(uid)
    Xz = zscore(X, stats)[:, keep].astype(np.float32)
    idx = np.arange(WINDOW, len(Xz), stride)
    A = np.full((len(idx), Xz.shape[1]), np.nan, np.float32)
    for b0 in range(0, len(idx), batch):
        ii = idx[b0:b0+batch]
        w = np.stack([Xz[i-WINDOW:i].T for i in ii]); y = Xz[ii]
        wt = torch.from_numpy(np.nan_to_num(w)).to(DEV)
        owt = torch.from_numpy(np.isfinite(w).astype(np.float32)).to(DEV)
        mu, ls = model(wt, owt, meta, context_vector(wt, ds_ref.ctx_idx, DEV), sensor_mask=None)
        a = np.abs(np.nan_to_num(y) - mu.cpu().numpy()) / (np.exp(ls.cpu().numpy()) + 1e-3)
        A[b0:b0+len(ii)] = np.where(np.isfinite(y), a, np.nan)
    return dict(uid=uid, a=A, t=t[idx], train=tt[idx], status=st[idx],
                score_cols=np.where(~SCHEMA[farm]["context"][keep].astype(bool))[0])

In [38]:
# ============ CELL 16: CONFORMAL EVIDENCE ============
def calibrate(a_cal):
    return [np.sort(a_cal[np.isfinite(a_cal[:,j]), j]) for j in range(a_cal.shape[1])]

def conformal_p(a, cal):
    n, S = a.shape; P = np.ones((n,S), np.float32)
    for j in range(S):
        c = cal[j]
        if len(c) < 50: continue
        P[:,j] = (len(c) - np.searchsorted(c, a[:,j], side="left") + 1) / (len(c) + 1)
    return np.clip(np.nan_to_num(P, nan=1.0), 1e-6, 1.0)

def e_process(P, k=0.4, subsample=6, floor=-20.0, cap=np.log(1e12)):
    n, S = P.shape
    logE = np.zeros((n,S), np.float32); acc = np.zeros(S, np.float32)
    for i in range(n):
        if i % subsample == 0:
            acc = np.clip(acc + np.log(k) + (k-1)*np.log(P[i]), floor, cap)
        logE[i] = acc
    mx = logE.max(1)
    return logE, np.log(np.exp(logE - mx[:,None]).mean(1) + 1e-30) + mx

def alarm_times(logEg, alpha=0.01):
    thr = math.log(1.0/alpha); hit = np.flatnonzero(logEg >= thr)
    return (int(hit[0]) if len(hit) else None), thr

In [39]:
# ============ CELL 17: EVAL ============
def evaluate_event(sc, ev_row, cal, alpha=0.01, k=0.4):
    pred = ~sc["train"].astype(bool)
    P = conformal_p(sc["a"][pred][:, sc["score_cols"]], cal)
    logE, logEg = e_process(P, k=k)
    ti, thr = alarm_times(logEg, alpha)
    t_pred = pd.to_datetime(sc["t"][pred], unit="s")
    dur_h = max((t_pred[-1]-t_pred[0]).total_seconds()/3600, 1)
    lead = (ev_row.start - t_pred[ti]).total_seconds()/3600 if (ti is not None and pd.notna(ev_row.start)) else np.nan
    return dict(uid=sc["uid"], is_anomaly=int(ev_row.is_anomaly), alarm=ti is not None,
                t_alarm=None if ti is None else t_pred[ti], lead_hours=lead,
                lead_frac=float(np.clip(lead/dur_h, 0, 1)) if pd.notna(lead) else 0.0,
                n_false_alarm=int(ev_row.is_anomaly==0 and ti is not None),
                turbine_months=dur_h/730.0, logEg=logEg, logE=logE)

def event_metrics(results, beta=0.5):
    R = pd.DataFrame(results)
    anom, norm = R[R.is_anomaly==1], R[R.is_anomaly==0]
    tp = int(anom.alarm.sum()); fn = len(anom)-tp
    fp = int(norm.alarm.sum()); tn = len(norm)-fp
    prec, rec = tp/max(tp+fp,1), tp/max(tp+fn,1)
    fb = (1+beta**2)*prec*rec/max(beta**2*prec+rec, 1e-9)
    early = float(anom.loc[anom.alarm, "lead_frac"].mean()) if tp else 0.0
    rel = tn/max(len(norm),1)
    return dict(CARE_approx=float(np.mean([rec, fb, rel, early])), Coverage=rec, Accuracy_F05=fb,
                Reliability=rel, Earliness=early, TP=tp, FP=fp, FN=fn, TN=tn,
                FA_per_turbine_month=float(norm.n_false_alarm.sum()/max(norm.turbine_months.sum(),1e-9)))

def run_fold(fold, epochs=6, alpha=0.01, cal_weeks=None, **train_kw):
    farm = fold["target_farm"]
    src = [u for u in fold["source"] if (NPZ/f"{u}.npz").exists()]
    tgt = [u for u in fold["target"] if (NPZ/f"{u}.npz").exists()]
    stats = fit_norm_stats(src, farm)
    model, ds, meta = train_sagewt(src, farm, stats, epochs=epochs, verbose=True, **train_kw)
    cal_rows = []
    for u in tgt:
        sc = score_event(model, u, farm, stats, ds, meta, stride=3)
        H = sc["train"].astype(bool) & np.isin(sc["status"], list(NORMAL_STATUS))
        A = sc["a"][H][:, sc["score_cols"]]
        if cal_weeks: A = A[-int(cal_weeks*7*24*60/RESOLUTION_MIN/3):]
        if len(A): cal_rows.append(A)
    cal = calibrate(np.concatenate(cal_rows, 0))
    res = [evaluate_event(score_event(model,u,farm,stats,ds,meta,stride=1),
                          EVENTS[EVENTS.uid==u].iloc[0], cal, alpha=alpha) for u in tgt]
    m = event_metrics(res); m["fold"] = fold["name"]
    return m, res

In [41]:
# ============ CELL 11: DATASET (preloaded) ============
class WindowSet(Dataset):
    def __init__(self, uids, farm, stats, window=WINDOW, stride=6, healthy_only=True, max_ram_gb=6):
        self.farm, self.st, self.L = farm, stats, window
        self.keep = np.where(stats["keep"])[0]
        n_ch = len(self.keep)
        self.store, self.items = {}, []
        est = 0
        for uid in uids:
            X, t, tt, s = load_cached(uid)
            m = (healthy_mask(tt, s) if healthy_only else np.ones(len(X), bool))
            m &= np.isfinite(X[:, self.keep]).mean(1) > 0.5
            Xz = zscore(X, stats)[:, self.keep].astype(np.float32)
            est += Xz.nbytes
            if est > max_ram_gb * 1e9:
                raise MemoryError(f"{est/1e9:.1f} GB > max_ram_gb; raise it or increase stride")
            self.store[uid] = Xz
            run = np.concatenate([[0], np.cumsum(m)])
            for i in range(window, len(X), stride):
                if run[i+1] - run[i-window] == window + 1: self.items.append((uid, i))
        ctx = SCHEMA[farm]["context"][self.keep].astype(bool)
        self.ctx_idx, self.score_idx = np.where(ctx)[0], np.where(~ctx)[0]
        print(f"  WindowSet: {len(self.items)} windows, {len(uids)} events, {n_ch} ch, {est/1e9:.2f} GB RAM")

    def __len__(self): return len(self.items)

    def __getitem__(self, k):
        uid, i = self.items[k]
        Xz = self.store[uid]
        w, y = Xz[i-self.L:i].T, Xz[i]
        return (torch.from_numpy(np.ascontiguousarray(np.nan_to_num(w))),
                torch.from_numpy(np.isfinite(w).astype(np.float32)),
                torch.from_numpy(np.nan_to_num(y)),
                torch.from_numpy(np.isfinite(y).astype(np.float32)))

In [42]:
# ============ CELL 14: TRAIN (AMP + timing) ============
from time import time
def train_sagewt(source_uids, farm, stats, epochs=8, bs=64, lr=3e-4, stride=6,
                 p_sensor=0.25, p_time=0.25, kill_meta=False, kill_film=False, verbose=True):
    ds = WindowSet(source_uids, farm, stats, stride=stride)
    dl = DataLoader(ds, batch_size=bs, shuffle=True, drop_last=True,
                    num_workers=0, pin_memory=(DEV=="cuda"))
    meta = {k: v.to(DEV) for k,v in meta_tensors(farm, np.where(stats["keep"])[0]).items()}
    model = SAGEWT().to(DEV)
    if kill_meta:
        for p in model.meta.parameters(): p.requires_grad_(False); p.data.zero_()
    if kill_film:
        for p in model.film.parameters(): p.requires_grad_(False); p.data.zero_()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, lr, epochs*max(len(dl),1))
    scaler = torch.cuda.amp.GradScaler(enabled=(DEV=="cuda"))
    for ep in range(epochs):
        model.train(); tot=nb=0; t0=time()
        for w, ow, y, oy in dl:
            w,ow,y,oy = [t.to(DEV, non_blocking=True) for t in (w,ow,y,oy)]
            B,S,L = w.shape
            sm, tm = make_masks(B,S,L,p_sensor,p_time,DEV)
            ctx = context_vector(w, ds.ctx_idx, DEV)
            with torch.cuda.amp.autocast(enabled=(DEV=="cuda")):
                mu, ls = model(w*(1-tm), ow*(1-tm), meta, ctx, sensor_mask=sm)
                loss = gnll(mu.float(), ls.float(), y, oy, 1.0+2.0*sm)
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
            tot += loss.item(); nb += 1
            if verbose and nb % 100 == 0:
                el = time()-t0
                print(f"    {nb}/{len(dl)} nll={tot/nb:.4f} {el:.0f}s "
                      f"(eta {el/nb*(len(dl)-nb):.0f}s)", end="\r")
        if verbose: print(f"  ep{ep+1}/{epochs} nll={tot/max(nb,1):.4f} [{time()-t0:.0f}s]")
    return model, ds, meta

In [43]:
fold = [f for f in SPLITS["cross_asset"] if f["target_farm"]=="A"][0]
m, r = run_fold(fold, epochs=3, stride=24)     # ~33k windows -> ~510 batches/epoch
print({k:(round(v,3) if isinstance(v,float) else v) for k,v in m.items() if k!="fold"})

  WindowSet: 32719 windows, 17 events, 56 ch, 0.21 GB RAM
  ep1/3 nll=184598.5315 [87s]5s (eta 2s))
  ep2/3 nll=11.3470 [87s]5s (eta 2s))
  ep3/3 nll=3.8025 [89s]8s (eta 2s))
{'CARE_approx': 0.486, 'Coverage': 0.667, 'Accuracy_F05': 0.667, 'Reliability': 0.5, 'Earliness': 0.109, 'TP': 2, 'FP': 1, 'FN': 1, 'TN': 1, 'FA_per_turbine_month': 0.849}


In [44]:
stats = fit_norm_stats(fold["source"], "A")
keep = np.where(stats["keep"])[0]
z = []
for u in fold["source"][:6]:
    X,t,tt,st = load_cached(u)
    Xz = zscore(X, stats)[:, keep]
    z.append(Xz[healthy_mask(tt,st)])
Z = np.concatenate(z,0)
q = np.nanpercentile(np.abs(Z), [50,99,99.9,100], axis=0)
print(f"|z| median={np.nanmedian(q[0]):.2f}  p99={np.nanmedian(q[1]):.2f}  "
      f"max overall={np.nanmax(q[3]):.1f}")
bad = np.argsort(-q[3])[:10]
for j in bad:
    print(f"  {SCHEMA['A']['names'][keep[j]]:22s} max|z|={q[3][j]:9.1f} p99={q[1][j]:6.2f} "
          f"q={QUANTS[SCHEMA['A']['quant'][keep[j]]]}")

|z| median=0.73  p99=2.21  max overall=1004.9
  sensor_26_avg          max|z|=   1004.9 p99=995.03 q=frequency
  sensor_51_avg          max|z|=     13.5 p99=  1.58 q=energy_reactive
  sensor_31_avg          max|z|=     13.4 p99=  1.58 q=power_reactive
  sensor_48_avg          max|z|=     13.0 p99=  1.56 q=energy_reactive
  sensor_13_avg          max|z|=     12.4 p99=  2.46 q=temperature
  sensor_32_avg          max|z|=     11.4 p99=  1.70 q=voltage
  sensor_33_avg          max|z|=     10.5 p99=  1.61 q=voltage
  sensor_14_avg          max|z|=      9.6 p99=  2.60 q=temperature
  sensor_34_avg          max|z|=      8.9 p99=  1.90 q=voltage
  sensor_47_avg          max|z|=      7.7 p99=  3.93 q=energy_reactive


In [45]:
# ============ CELL 9b: ROBUST NORM STATS ============
def fit_norm_stats(uids, farm, clip=8.0, robust=True):
    S = len(SCHEMA[farm]["names"]); acc = [[] for _ in range(S)]
    for uid in uids:
        X,t,tt,st = load_cached(uid)
        Xi = X[healthy_mask(tt,st)]
        if len(Xi): acc_i = Xi[::7]; [acc[j].append(acc_i[:,j]) for j in range(S)]
    mu = np.zeros(S,np.float32); sd = np.ones(S,np.float32); n = np.zeros(S)
    for j in range(S):
        v = np.concatenate(acc[j]) if acc[j] else np.array([])
        v = v[np.isfinite(v)]; n[j] = len(v)
        if len(v) < 500: continue
        if robust:
            med = np.median(v); iqr = np.subtract(*np.percentile(v,[75,25]))
            mu[j] = med; sd[j] = max(iqr/1.349, 1e-3*max(abs(med),1.0), 1e-6)
        else:
            mu[j] = v.mean(); sd[j] = max(v.std(), 1e-6)
    return dict(mu=mu, sd=sd, keep=(n>500)&(sd>1e-6), clip=clip,
                nan_rate=np.zeros(S,np.float32))

def zscore(X, st):
    Z = (X - st["mu"]) / st["sd"]
    c = st.get("clip")
    return np.clip(Z, -c, c) if c else Z

In [47]:
j = SCHEMA["A"]["names"].index("sensor_26_avg")
print(FEAT[(FEAT.farm=="A") & (FEAT.name=="sensor_26_avg")][["description","unit_n","quantity"]].to_string(index=False))
u = fold["source"][0]
X,t,tt,st = load_cached(u); x = X[healthy_mask(tt,st), j]; x = x[np.isfinite(x)]
print(f"n={len(x)} min={x.min():.4g} max={x.max():.4g} med={np.median(x):.4g} "
      f"IQR={np.subtract(*np.percentile(x,[75,25])):.4g}")
print("percentiles:", np.percentile(x,[0,1,25,50,75,99,100]).round(4))
print("n_unique:", len(np.unique(x)), "| first 20 raw:", x[:20].round(4))

   description unit_n  quantity
Grid frequency     hz frequency
n=52063 min=49.9 max=50.1 med=50 IQR=0
percentiles: [49.9 50.  50.  50.  50.  50.  50.1]
n_unique: 3 | first 20 raw: [50. 50. 50. 50. 50. 50. 50. 50. 50. 50. 50. 50. 50. 50. 50. 50. 50. 50.
 50. 50.]


In [48]:
# ============ CELL 9c: NORM STATS WITH DEGENERATE-SCALE GUARD ============
def fit_norm_stats(uids, farm, clip=8.0, min_rel_iqr=1e-2):
    S = len(SCHEMA[farm]["names"]); acc = [[] for _ in range(S)]
    for uid in uids:
        X,t,tt,st = load_cached(uid)
        Xi = X[healthy_mask(tt,st)]
        if len(Xi):
            sub = Xi[::7]
            for j in range(S): acc[j].append(sub[:,j])
    mu = np.zeros(S,np.float32); sd = np.ones(S,np.float32)
    n = np.zeros(S); degen = np.zeros(S, bool)
    for j in range(S):
        v = np.concatenate(acc[j]) if acc[j] else np.array([])
        v = v[np.isfinite(v)]; n[j] = len(v)
        if len(v) < 500: continue
        med = float(np.median(v)); iqr = float(np.subtract(*np.percentile(v,[75,25])))
        scale = max(abs(med), 1.0)
        if iqr / scale < min_rel_iqr:          # near-constant relative to its own level
            degen[j] = True
            iqr = max(iqr, min_rel_iqr * scale)   # cap amplification instead of exploding
        mu[j] = med; sd[j] = max(iqr/1.349, 1e-6)
    keep = (n > 500) & (sd > 1e-6)
    print(f"  [{farm}] keep={int(keep.sum())}/{S} | degenerate-scale={int(degen.sum())}: "
          f"{[SCHEMA[farm]['names'][j] for j in np.where(degen)[0]][:6]}")
    return dict(mu=mu, sd=sd, keep=keep, degen=degen, clip=clip, nan_rate=np.zeros(S,np.float32))

In [ ]:
res_A = []
for f in [x for x in SPLITS["cross_asset"] if x["target_farm"]=="A"]:
    m, r = run_fold(f, epochs=6, stride=12)
    res_A.append(m); print(f"{f['name']:18s} CARE~{m['CARE_approx']:.3f} "
                           f"cov={m['Coverage']:.2f} rel={m['Reliability']:.2f} "
                           f"FA/mo={m['FA_per_turbine_month']:.2f}")
print(pd.DataFrame(res_A).drop(columns=["fold"]).mean().round(3))

  [A] keep=56/58 | degenerate-scale=4: ['sensor_26_avg', 'sensor_32_avg', 'sensor_33_avg', 'sensor_34_avg']
  WindowSet: 65429 windows, 17 events, 56 ch, 0.21 GB RAM
